# Pinehill Unified School District — Data Cleaning & Analysis

**End-to-end pipeline:** raw multi-source district data → cleaning, validation & reconciliation → exploratory and statistical analysis → Tableau-ready exports.

**Author:** Jared Oberg | **Tools:** Python (pandas, scikit-learn, scipy, matplotlib/seaborn) → Tableau

> **Note:** All data is fictional, created for portfolio purposes. Pinehill USD does not exist.

**Notebook contents**
1. Setup & Data Loading
2. Site ID Reconciliation
3. Cleaning & Feature Engineering
4. Data Quality Report
5. Assessment Scores — Exploratory Analysis
6. Demographics & Attendance by School
7. Staffing & Spending Aggregates
8. Master Table, Correlations & Regression
9. Statistical Significance Testing
10. Funding Equity Analysis
11. Professional Development vs. Turnover
12. Cohort Reading Growth
13. Export for Tableau
14. Limitations & Conclusions

## 1. Setup & Data Loading

The district's data lives in five disconnected systems, each with its own format:

| Source | Format | Contents |
|---|---|---|
| SIS extracts | Pipe-delimited `.txt` | Student demographics, attendance |
| Assessment platform | Nested JSON | ELA/Math proficiency by school, year, grade, subgroup |
| Escape Online (finance) | CSV | GL budget/actuals by school, year, category |
| Escape Online (HR) | CSV | Positions, FTE, turnover, days-to-fill |
| Reference files | CSV | School list, site crosswalk, code legend |

Raw files are expected in `data/raw/`; cleaned Tableau exports are written to `data/clean/`.

In [ ]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)

DATA_DIR = os.path.join('data', 'raw')
OUTPUT_DIR = os.path.join('data', 'clean')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Reference files ──
schools_ref = pd.read_csv(os.path.join(DATA_DIR, 'schools_reference.csv'))
crosswalk   = pd.read_csv(os.path.join(DATA_DIR, 'site_crosswalk.csv'))
code_legend = pd.read_csv(os.path.join(DATA_DIR, 'code_legend.csv'))

# ── Academic files ──
cohort = pd.read_csv(os.path.join(DATA_DIR, 'cohort_reading_longitudinal.csv'))
grades = pd.read_csv(os.path.join(DATA_DIR, 'course_grades.csv'))

# ── Assessment JSON (nested → flat) ──
with open(os.path.join(DATA_DIR, 'assessment_results.json')) as f:
    raw = json.load(f)
assessment = pd.json_normalize(raw['data'])
assessment = assessment.rename(columns={
    'assessments.ela.pct_proficient':  'ela_pct_proficient',
    'assessments.math.pct_proficient': 'math_pct_proficient',
    'assessments.ela.tested':          'ela_tested',
    'assessments.math.tested':         'math_tested',
    'school':                          'school_name'
})
assessment = assessment.drop(
    columns=[c for c in assessment.columns if 'assessments.' in c])

# ── SIS extracts (pipe-delimited, header row to skip) ──
stu = pd.read_csv(os.path.join(DATA_DIR, 'SIS_STU_20260602.txt'), sep='|', skiprows=1)
att = pd.read_csv(os.path.join(DATA_DIR, 'SIS_ATT_20260602.txt'), sep='|', skiprows=1)
stu.columns = stu.columns.str.strip()
att.columns = att.columns.str.strip()

# ── Financial / HR files ──
gl = pd.read_csv(os.path.join(DATA_DIR, 'ESCAPEONLINE_GL_20260602.csv'), skiprows=1)
gl.columns = gl.columns.str.strip()
positions = pd.read_csv(os.path.join(DATA_DIR, 'ESCAPEONLINE_POSITIONS_20260602.csv'), skiprows=1)
positions.columns = positions.columns.str.strip()

print("=== All Files Loaded ===")
for name, df in [('schools_ref', schools_ref), ('assessment', assessment),
                 ('cohort', cohort), ('grades', grades), ('stu', stu),
                 ('att', att), ('gl', gl), ('positions', positions)]:
    print(f"{name:12s} {df.shape}")

## 2. Site ID Reconciliation

The three source systems identify the same 12 schools three different ways — SIS uses codes 1–12, Escape Online uses 10–140, and the reference files use `PH-01`–`PH-12`. **The provided crosswalk file was incomplete: Dogwood Elementary (PH-04) was missing entirely**, so the mapping below was reconciled manually and validated against enrollment counts in each system.

In [ ]:
school_map_sis = {
    1: 'Aspen Elementary',   2: 'Birch Elementary',
    3: 'Cedar Elementary',   4: 'Dogwood Elementary',
    5: 'Elm Elementary',     6: 'Fir Elementary',
    7: 'Granite Elementary', 8: 'Hawthorn Elementary',
    9: 'Junction Middle School', 10: 'Keystone Middle School',
    11: 'Northgate High School', 12: 'Summit High School'
}
school_map_gl = {
    10: 'Aspen Elementary',   20: 'Birch Elementary',
    30: 'Cedar Elementary',   40: 'Dogwood Elementary',
    50: 'Elm Elementary',     60: 'Fir Elementary',
    70: 'Granite Elementary', 80: 'Hawthorn Elementary',
    110: 'Junction Middle School', 120: 'Keystone Middle School',
    130: 'Northgate High School',  140: 'Summit High School'
}
print("Site ID mappings defined (SIS 1-12, Escape 10-140).")

## 3. Cleaning & Feature Engineering

**Decoding & merging.** SIS categorical codes (`EL_STAT`, `FRL`, `RACE1`) are decoded to labels; student and attendance extracts are merged on `PERM_ID`.

**Engineered features:**
- `absence_rate` — days absent / days enrolled
- `chronically_absent` — flag at the standard ≥10% threshold
- `SED` — socioeconomically disadvantaged (Free or Reduced lunch)

**Corrections applied:**
- One duplicate record removed from the grades dataset (Junction MS, 2024-25 Math)
- **Elm Elementary 2026 Books & Supplies data-entry error**: actuals of $2.13M against a $224K budget (952% spend rate — a misplaced decimal). Flagged and imputed at the district-wide 95.24% spend rate rather than dropped, preserving the school's row for cross-school comparisons.

In [ ]:
# ── Decode SIS categorical fields ──
for col in ['EL_STAT', 'FRL', 'SPED', 'HISP']:
    stu[col] = stu[col].str.strip()

stu['EL_label'] = stu['EL_STAT'].map({
    'EO': 'English Only', 'IFEP': 'Initially Fluent',
    'EL': 'English Learner', 'RFEP': 'Reclassified Fluent'
})
stu['FRL_label'] = stu['FRL'].map({'F': 'Free', 'R': 'Reduced', 'N': 'Not Eligible'})
stu['RACE_label'] = stu['RACE1'].map({
    100: 'Hispanic or Latino', 300: 'Asian',
    400: 'Black or African American', 700: 'White',
    800: 'Two or More Races'
})
stu['SED'] = stu['FRL'].isin(['F', 'R'])
stu['school_name'] = stu['SC'].map(school_map_sis)
att['school_name'] = att['SC'].map(school_map_sis)

# ── Merge student + attendance ──
merged = stu.merge(att, on='PERM_ID', suffixes=('_stu', '_att'))
merged['school_name'] = merged['SC_stu'].map(school_map_sis)
merged['absence_rate'] = (merged['DAYS_ABSENT'] / merged['DAYS_ENROLLED'] * 100).round(1)
merged['chronically_absent'] = merged['absence_rate'] >= 10

print(f"Merged student records: {merged.shape}")
print("Chronic absence threshold: >= 10% of enrolled days")

In [ ]:
# ── Grades: remove duplicate record ──
grades_clean = grades.drop_duplicates()
print(f"Grades: {len(grades)} -> {len(grades_clean)} rows after dedup")

# ── GL: map schools/resources, fix Elm 2026 Books & Supplies error ──
gl['school_name'] = gl['Location'].map(school_map_gl)
resource_map = {0: 'General Fund', 3010: 'Title I Federal', 7510: 'LCFF Supplemental'}
gl['resource_label'] = gl['Resource'].map(resource_map)

elm_mask = (
    (gl['school_name'] == 'Elm Elementary') &
    (gl['FiscalYear'] == 2026) &
    (gl['AccountDescription'] == 'Books & Supplies')
)
gl['data_flag'] = ''
gl.loc[elm_mask, 'data_flag'] = 'DATA_ENTRY_ERROR'

print("\nElm 2026 Books & Supplies before correction:")
print(gl.loc[elm_mask, ['Budget', 'Actuals']].to_string(index=False))

# Impute actuals at the district-wide 95.24% spend rate
gl_final = gl.copy()
elm_budget = gl.loc[elm_mask, 'Budget'].values[0]
gl_final.loc[elm_mask, 'Actuals'] = (elm_budget * 0.952381).round(2)
gl_final.loc[elm_mask, 'data_flag'] = 'IMPUTED'

print("\nAfter imputation:")
print(gl_final.loc[elm_mask, ['Budget', 'Actuals', 'data_flag']].to_string(index=False))

## 4. Data Quality Report

A consolidated audit of every source file: duplicates, missing values, and the full list of issues discovered during validation.

In [ ]:
print("=" * 60)
print("DATA QUALITY REPORT — PINEHILL USD")
print("=" * 60)

files_to_check = {
    'schools_ref': schools_ref, 'assessment': assessment,
    'cohort': cohort, 'grades': grades,
    'stu': stu, 'att': att, 'gl': gl
}

print("\n--- DUPLICATES ---")
for name, df in files_to_check.items():
    dups = df.duplicated().sum()
    flag = 'FOUND' if dups > 0 else 'None'
    print(f"  {name:20s}: {flag} ({dups})")

print("\n--- MISSING VALUES ---")
for name, df in files_to_check.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"\n  {name}:")
        for col, count in nulls.items():
            print(f"    {col}: {count} ({count/len(df)*100:.1f}%)")
    else:
        print(f"  {name:20s}: No missing values")

print("\n--- KNOWN ISSUES (discovered & handled) ---")
issues = [
    "Crosswalk missing Dogwood Elementary (PH-04) — manually reconciled",
    "Elm 2026 Books & Supplies $2.1M vs $224K budget — flagged & imputed",
    "Grades: 1 duplicate removed (Junction MS 2024-25 Math)",
    "STU: 290 students (3.8%) with undocumented race codes (200, 500, 600)",
    "Three site ID formats: SIS=1-12, Escape=10-140, Ref=PH-01..12",
    "Three date formats: SIS=CCYYMMDD, CSV=2025-26, GL=integer year",
    "All schools show identical 4.8% unspent budget (synthetic-data artifact)",
]
for i, issue in enumerate(issues, 1):
    print(f"  {i}. {issue}")

## 5. Assessment Scores — Exploratory Analysis

Average proficiency by school (ALL subgroup), year-over-year trends, and an IQR-based outlier scan across subgroups. Birch Elementary consistently leads in both subjects; Granite Elementary consistently ranks last and becomes a focus school for the rest of the analysis.

In [ ]:
# ── Average scores per school (ALL subgroup) ──
scores_avg = assessment[assessment['subgroup'] == 'ALL']\
    .groupby('school_name')[['ela_pct_proficient', 'math_pct_proficient']]\
    .mean().round(1).reset_index()
scores_avg['avg_score'] = (
    (scores_avg['ela_pct_proficient'] + scores_avg['math_pct_proficient']) / 2
).round(1)

print("=== AVERAGE PROFICIENCY BY SCHOOL ===")
print(scores_avg.sort_values('avg_score', ascending=False).to_string(index=False))

In [ ]:
# ── Year-over-year trends ──
scores = assessment[assessment['subgroup'] == 'ALL']\
    .groupby(['school_name', 'school_year'])[['ela_pct_proficient', 'math_pct_proficient']]\
    .mean().round(1).reset_index()

ela_pivot = scores.pivot(index='school_name', columns='school_year', values='ela_pct_proficient')
ela_pivot.columns = [f'ELA_{yr}' for yr in ela_pivot.columns]
math_pivot = scores.pivot(index='school_name', columns='school_year', values='math_pct_proficient')
math_pivot.columns = [f'MATH_{yr}' for yr in math_pivot.columns]
combined = pd.concat([ela_pivot, math_pivot], axis=1)

years = sorted(assessment['school_year'].unique())
combined['ELA_AVG'] = combined[[f'ELA_{yr}' for yr in years]].mean(axis=1).round(1)
combined['MATH_AVG'] = combined[[f'MATH_{yr}' for yr in years]].mean(axis=1).round(1)
combined['ELA_TREND'] = (combined[f'ELA_{years[-1]}'] - combined[f'ELA_{years[0]}']).round(1)
combined['MATH_TREND'] = (combined[f'MATH_{years[-1]}'] - combined[f'MATH_{years[0]}']).round(1)

print("=== ELA SCORES & TREND (first year -> last year) ===")
ela_cols = [f'ELA_{yr}' for yr in years] + ['ELA_AVG', 'ELA_TREND']
print(combined[ela_cols].sort_values('ELA_AVG', ascending=False).to_string())

print("\n=== MATH SCORES & TREND ===")
math_cols = [f'MATH_{yr}' for yr in years] + ['MATH_AVG', 'MATH_TREND']
print(combined[math_cols].sort_values('MATH_AVG', ascending=False).to_string())

In [ ]:
# ── Outlier scan (IQR method) across subgroups & years ──
outliers = []
for subgroup in ['ALL', 'EL', 'SED', 'SWD']:
    for year in assessment['school_year'].unique():
        subset = assessment[
            (assessment['subgroup'] == subgroup) &
            (assessment['school_year'] == year)
        ]
        for col in ['ela_pct_proficient', 'math_pct_proficient']:
            Q1, Q3 = subset[col].quantile(0.25), subset[col].quantile(0.75)
            IQR = Q3 - Q1
            lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            for _, row in subset.iterrows():
                if row[col] < lower or row[col] > upper:
                    outliers.append({
                        'year': year, 'subgroup': subgroup, 'subject': col,
                        'school': row['school_name'], 'score': row[col],
                        'direction': 'HIGH' if row[col] > upper else 'LOW'
                    })

outliers_df = pd.DataFrame(outliers)
print("=== STATISTICAL OUTLIERS (IQR METHOD) ===")
if len(outliers_df):
    print(outliers_df.groupby(['school', 'direction']).size()
          .reset_index(name='count')
          .sort_values('count', ascending=False).to_string(index=False))
else:
    print("No outliers detected.")

## 6. Demographics & Attendance by School

School-level aggregation of the merged student data. Note the chronic-absence rate is computed as **chronic count / total students** — an earlier version of this calculation was audited and corrected during development.

In [ ]:
# ── Demographics ──
demo = merged.groupby('school_name').agg(
    total_students=('PERM_ID', 'count'),
    pct_free=('FRL', lambda x: (x == 'F').mean() * 100),
    pct_reduced=('FRL', lambda x: (x == 'R').mean() * 100),
    pct_sed=('SED', lambda x: x.mean() * 100),
    pct_el=('EL_STAT', lambda x: (x == 'EL').mean() * 100),
    pct_sped=('SPED', lambda x: (x == 'Y').mean() * 100),
    pct_hisp=('HISP', lambda x: (x == 'Y').mean() * 100),
).round(1).reset_index()

print("=== DEMOGRAPHICS BY SCHOOL (sorted by SED%) ===")
print(demo.sort_values('pct_sed', ascending=False).to_string(index=False))

In [ ]:
# ── Attendance (verified calculation) ──
attend_school = merged.groupby('school_name').agg(
    avg_absence_rate=('absence_rate', 'mean'),
    avg_tardies=('TARDIES', 'mean'),
    chronic_count=('chronically_absent', 'sum'),
    total_students=('PERM_ID', 'count')
).reset_index()
attend_school['pct_chronically_absent'] = (
    attend_school['chronic_count'] / attend_school['total_students'] * 100
).round(1)
attend_school = attend_school.drop(columns=['chronic_count', 'total_students']).round(1)

print("=== ATTENDANCE BY SCHOOL ===")
print(attend_school.sort_values('pct_chronically_absent', ascending=False)
      .to_string(index=False))

## 7. Staffing & Spending Aggregates

Multi-year averages from the HR positions file (turnover, vacancies, days-to-fill) and per-pupil spending by GL category, combined into a single analysis table (`spend_corr`).

In [ ]:
# ── Staffing averages per school ──
positions_named = positions.rename(columns={'SiteName': 'school_name'})
positions_avg = positions_named.groupby('school_name').agg(
    avg_filled_fte=('FilledFTE', 'mean'),
    avg_vacant_fte=('VacantFTE', 'mean'),
    avg_turnover=('TeacherTurnoverPct', 'mean'),
    avg_days_to_fill=('AvgDaysToFillVacancy', 'mean'),
).round(1).reset_index()

positions_avg = positions_avg.merge(
    schools_ref[['school_name', 'enrollment_2025_26']], on='school_name')
positions_avg['ratio_filled'] = (
    positions_avg['enrollment_2025_26'] / positions_avg['avg_filled_fte']).round(1)
positions_avg['vacant_per_100'] = (
    positions_avg['avg_vacant_fte'] / positions_avg['enrollment_2025_26'] * 100).round(2)

print("=== STAFFING AVERAGES BY SCHOOL ===")
print(positions_avg.sort_values('avg_turnover', ascending=False).to_string(index=False))

In [ ]:
# ── Per-pupil spending by category ──
spend_pp = gl_final.groupby(['school_name', 'AccountDescription'])['Actuals']\
    .mean().reset_index()
spend_pp = spend_pp.merge(
    schools_ref[['school_name', 'enrollment_2025_26']], on='school_name')
spend_pp['actuals_pp'] = (spend_pp['Actuals'] / spend_pp['enrollment_2025_26']).round(0)

spend_pivot = spend_pp.pivot(index='school_name', columns='AccountDescription',
                             values='actuals_pp').reset_index()
spend_pivot.columns.name = None

# Convenience alias for the professional development category
prof_dev_col = [c for c in spend_pivot.columns
                if 'prof' in c.lower() or 'contracted' in c.lower()][0]
spend_pivot['ProfDev_pp'] = spend_pivot[prof_dev_col]

# ── Combined analysis table: spending + staffing + attendance + scores ──
spend_corr = spend_pivot.merge(
    positions_avg[['school_name', 'avg_turnover', 'avg_days_to_fill',
                   'vacant_per_100']], on='school_name')
spend_corr = spend_corr.merge(attend_school, on='school_name')
spend_corr = spend_corr.merge(scores_avg, on='school_name')

print(f"spend_corr: {spend_corr.shape}")
print(f"GL categories: {[c for c in spend_pivot.columns if c not in ('school_name', 'ProfDev_pp')]}")

## 8. Master Table, Correlations & Regression

One row per school combining scores, demographics, and attendance — the base table for all correlation work. With **n = 12 schools**, correlations here are directional signals, not precise estimates (see Section 14).

In [ ]:
# ── Master table ──
master = scores_avg.merge(demo, on='school_name')
master = master.merge(attend_school, on='school_name')

master_table = master.sort_values('avg_score', ascending=False).reset_index(drop=True)
master_table['rank'] = master_table.index + 1

numeric_cols = [c for c in master.columns if c not in ('school_name', 'total_students')]
corr = master[numeric_cols].corr().round(2)

print("=== CORRELATION WITH ELA SCORE ===")
print(corr['ela_pct_proficient'].drop('ela_pct_proficient').sort_values().to_string())

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Matrix — Demographics & Attendance vs Scores\nPinehill USD',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Multiple regression: how much score variation do these factors explain? ──
reg_data = master.copy()
features = ['pct_el', 'avg_absence_rate', 'avg_tardies']
X = StandardScaler().fit_transform(reg_data[features])

for target in ['ela_pct_proficient', 'math_pct_proficient']:
    y = reg_data[target]
    reg = LinearRegression().fit(X, y)
    print(f"=== Regression: {target} ===")
    print(f"R^2: {reg.score(X, y):.3f}")
    coef_df = pd.DataFrame({'feature': features,
                            'coefficient': reg.coef_.round(3)})\
        .sort_values('coefficient', key=abs, ascending=False)
    print(coef_df.to_string(index=False), "\n")

## 9. Statistical Significance Testing

Pearson correlations with p-values for every key relationship in the analysis. With n = 12, statistical significance at p < 0.05 requires |r| ≳ 0.58 — a deliberately high bar that filters out weak school-level associations.

In [ ]:
# ── Build the full verification table (scores + demo + attendance + staffing + PD) ──
verify = scores_avg.merge(demo, on='school_name')
verify = verify.merge(attend_school, on='school_name')
verify = verify.merge(
    positions_avg[['school_name', 'avg_turnover', 'avg_days_to_fill',
                   'vacant_per_100']], on='school_name')
verify = verify.merge(spend_corr[['school_name', 'ProfDev_pp']], on='school_name')

tests = [
    (master['pct_el'],  master['ela_pct_proficient'],  'EL% -> ELA Score',          'Demographics->Scores'),
    (master['pct_el'],  master['math_pct_proficient'], 'EL% -> Math Score',         'Demographics->Scores'),
    (master['pct_free'], master['ela_pct_proficient'], 'Free Lunch% -> ELA Score',  'Demographics->Scores'),
    (master['pct_sed'], master['ela_pct_proficient'],  'SED% -> ELA Score',         'Demographics->Scores'),
    (master['avg_absence_rate'], master['ela_pct_proficient'], 'Absence Rate -> ELA', 'Attendance->Scores'),
    (master['avg_tardies'], master['ela_pct_proficient'], 'Tardies -> ELA',          'Attendance->Scores'),
    (master['pct_chronically_absent'], master['ela_pct_proficient'], 'Chronic Absent% -> ELA', 'Attendance->Scores'),
    (verify['avg_turnover'], verify['avg_absence_rate'], 'Turnover -> Absence Rate', 'Staffing->Attendance'),
    (verify['avg_turnover'], verify['pct_chronically_absent'], 'Turnover -> Chronic Absent', 'Staffing->Attendance'),
    (verify['vacant_per_100'], verify['avg_absence_rate'], 'Vacancies -> Absence Rate', 'Staffing->Attendance'),
    (spend_corr['ProfDev_pp'], spend_corr['avg_turnover'], 'Prof Dev -> Turnover',   'Budget->Staffing'),
    (spend_corr['ProfDev_pp'], spend_corr['ela_pct_proficient'], 'Prof Dev -> ELA Score', 'Budget->Scores'),
]

rows = []
print("=== STATISTICAL SIGNIFICANCE OF KEY CORRELATIONS (n=12 schools) ===\n")
print(f"{'Relationship':<32} {'r':>7} {'p-value':>9} {'Sig?':>10} {'Effect':>8}")
print("-" * 72)
for x, y, label, category in tests:
    mask = ~(x.isna() | y.isna())
    r, p = stats.pearsonr(x[mask], y[mask])
    effect = 'Large' if abs(r) >= 0.5 else 'Medium' if abs(r) >= 0.3 else 'Small'
    sig = 'Yes' if p < 0.05 else ('Marginal' if p < 0.10 else 'No')
    rows.append({'relationship': label, 'r_value': round(r, 3),
                 'p_value': round(p, 4), 'significant': sig,
                 'effect_size': effect, 'category': category})
    print(f"{label:<32} {r:>7.3f} {p:>9.4f} {sig:>10} {effect:>8}")

corr_summary = pd.DataFrame(rows)

In [ ]:
# ── Overall regression model significance (F-test) ──
X_reg = reg_data[features].values
n, p_feat = len(X_reg), len(features)

for target in ['ela_pct_proficient', 'math_pct_proficient']:
    y = reg_data[target].values
    reg = LinearRegression().fit(X_reg, y)
    r2 = reg.score(X_reg, y)
    f_stat = (r2 / p_feat) / ((1 - r2) / (n - p_feat - 1))
    p_f = 1 - stats.f.cdf(f_stat, p_feat, n - p_feat - 1)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p_feat - 1)
    print(f"{target}: R^2={r2:.3f} | Adj R^2={adj_r2:.3f} | "
          f"F({p_feat},{n-p_feat-1})={f_stat:.2f} | p={p_f:.4f} | "
          f"{'Significant' if p_f < 0.05 else 'Not significant'}")

## 10. Funding Equity Analysis

**Question:** Is Title I + LCFF intervention funding distributed in proportion to student need (EL or SED students)?

**Answer:** No. Funding per student in need varies roughly 3x across schools. The recommendation reallocates the *existing* funding pool proportionally to each school's count of students in need — budget-neutral by construction.

In [ ]:
# ── Students in need per school ──
need_students = merged.groupby('school_name').agg(
    total_students=('PERM_ID', 'count'),
    el_students=('EL_STAT', lambda x: (x == 'EL').sum()),
    sed_students=('SED', 'sum'),
).reset_index()
need_students['el_or_sed'] = merged.groupby('school_name').apply(
    lambda x: ((x['EL_STAT'] == 'EL') | (x['SED'])).sum()).values

# ── Intervention funding by resource ──
resource_total = gl_final.groupby(['school_name', 'resource_label'])['Actuals']\
    .sum().round(0).unstack(fill_value=0).reset_index()
resource_total.columns.name = None

funding_need = need_students.merge(resource_total, on='school_name')
funding_need = funding_need.merge(scores_avg, on='school_name')

funding_need['titleI_per_need'] = (
    funding_need['Title I Federal'] / funding_need['el_or_sed']).round(0)
funding_need['lcff_per_need'] = (
    funding_need['LCFF Supplemental'] / funding_need['el_or_sed']).round(0)
funding_need['total_per_need'] = (
    (funding_need['Title I Federal'] + funding_need['LCFF Supplemental'])
    / funding_need['el_or_sed']).round(0)

# ── Recommended need-proportional allocation (budget-neutral) ──
total_need = funding_need['el_or_sed'].sum()
total_titleI = funding_need['Title I Federal'].sum()
total_lcff = funding_need['LCFF Supplemental'].sum()
funding_need['recommended_titleI'] = (
    funding_need['el_or_sed'] / total_need * total_titleI).round(0)
funding_need['recommended_lcff'] = (
    funding_need['el_or_sed'] / total_need * total_lcff).round(0)
funding_need['recommended_titleI_per_need'] = (
    funding_need['recommended_titleI'] / funding_need['el_or_sed']).round(0)
funding_need['recommended_lcff_per_need'] = (
    funding_need['recommended_lcff'] / funding_need['el_or_sed']).round(0)

print("=== INTERVENTION FUNDING PER STUDENT IN NEED ===")
cols = ['school_name', 'el_or_sed', 'total_per_need', 'ela_pct_proficient']
print(funding_need[cols].sort_values('total_per_need', ascending=False)
      .to_string(index=False))
print(f"\nRange: ${funding_need['total_per_need'].min():,.0f} - "
      f"${funding_need['total_per_need'].max():,.0f} per student in need")

In [ ]:
# ── Chart: funding per student in need, schools ordered by need ──
funding_sorted = funding_need.sort_values('el_or_sed', ascending=False)\
    .reset_index(drop=True)
short_names = [s.replace(' Elementary', '\nElem')
                .replace(' Middle School', '\nMS')
                .replace(' High School', '\nHS')
               for s in funding_sorted['school_name']]
x = np.arange(len(funding_sorted))

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.bar(x, funding_sorted['total_per_need'],
              color='#FF9800', edgecolor='black', linewidth=0.5)

district_avg = ((funding_sorted['Title I Federal'] +
                 funding_sorted['LCFF Supplemental']).sum()
                / funding_sorted['el_or_sed'].sum())
ax.axhline(y=district_avg, color='#D32F2F', linestyle='--', linewidth=2,
           label=f'District Avg: ${district_avg:,.0f}')

for bar, val in zip(bars, funding_sorted['total_per_need']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 25,
            f'${int(val):,}', ha='center', va='bottom',
            fontsize=12, fontweight='bold')
for i, (need, total) in enumerate(zip(funding_sorted['el_or_sed'],
                                      funding_sorted['total_students'])):
    ax.text(i, -120, f'n={int(need)}\n({need/total*100:.0f}%)',
            ha='center', fontsize=9, color='#555555', fontweight='bold')

ax.set_title('Current Intervention Funding per Student in Need\n'
             'Schools ordered by students in need — highest to lowest',
             fontsize=15, fontweight='bold')
ax.set_ylabel('$ per Student in Need', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=12, fontweight='bold')
ax.set_ylim(-250, funding_sorted['total_per_need'].max() * 1.2)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart: current vs recommended total intervention funding ──
funding_reco = funding_sorted.copy()
funding_reco['current_total'] = (
    funding_reco['Title I Federal'] + funding_reco['LCFF Supplemental'])
funding_reco['recommended_total'] = (
    funding_reco['recommended_titleI'] + funding_reco['recommended_lcff'])

x = np.arange(len(funding_reco))
width = 0.35
fig, ax = plt.subplots(figsize=(16, 8))
ax.bar(x - width / 2, funding_reco['current_total'], width,
       label='Current Funding', color='#FF9800', edgecolor='black', linewidth=0.5)
ax.bar(x + width / 2, funding_reco['recommended_total'], width,
       label='Recommended Need-Based Funding', color='#4CAF50',
       edgecolor='black', linewidth=0.5)

for xi, (cur, rec) in enumerate(zip(funding_reco['current_total'],
                                    funding_reco['recommended_total'])):
    ax.text(xi - width / 2, cur + 3000, f'${cur/1000:.0f}K',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.text(xi + width / 2, rec + 3000, f'${rec/1000:.0f}K',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Current vs Recommended Intervention Funding\nTitle I + LCFF Combined',
             fontsize=17, fontweight='bold')
ax.set_ylabel('Total Intervention Funding ($)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=12, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.set_ylim(0, funding_reco['recommended_total'].max() * 1.2)
plt.tight_layout()
plt.show()

## 11. Professional Development vs. Turnover

Across all seven budget categories, professional development spending per pupil is the only one with a statistically significant relationship to teacher turnover (Section 9: r = −0.59, p < 0.05). The highest-turnover schools spend the least on PD.

In [ ]:
# ── Correlation of every spending category with turnover / attendance / scores ──
spend_cols = [c for c in spend_pivot.columns
              if c not in ('school_name', 'ProfDev_pp')]

print("=== SPENDING CATEGORIES (per pupil) vs TURNOVER, ATTENDANCE & SCORES ===\n")
print(f"{'Category':<32} {'Turnover':>9} {'Chronic%':>9} {'ELA':>7}")
print("-" * 60)
for col in spend_cols:
    r_turn = spend_corr[[col, 'avg_turnover']].corr().iloc[0, 1]
    r_chr = spend_corr[[col, 'pct_chronically_absent']].corr().iloc[0, 1]
    r_ela = spend_corr[[col, 'ela_pct_proficient']].corr().iloc[0, 1]
    print(f"{col:<32} {r_turn:>9.3f} {r_chr:>9.3f} {r_ela:>7.3f}")

In [ ]:
# ── Scatter: PD spending per pupil vs turnover ──
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2196F3' if 'Elementary' in s
          else '#F44336' if 'Middle' in s else '#FF9800'
          for s in spend_corr['school_name']]
ax.scatter(spend_corr['ProfDev_pp'], spend_corr['avg_turnover'],
           s=180, c=colors, edgecolor='black', linewidth=0.8, zorder=3)
for _, row in spend_corr.iterrows():
    ax.annotate(row['school_name'].split()[0],
                (row['ProfDev_pp'], row['avg_turnover']),
                textcoords='offset points', xytext=(8, 6), fontsize=10)

m, b = np.polyfit(spend_corr['ProfDev_pp'], spend_corr['avg_turnover'], 1)
xs = np.linspace(spend_corr['ProfDev_pp'].min(),
                 spend_corr['ProfDev_pp'].max(), 50)
ax.plot(xs, m * xs + b, '--', color='gray', linewidth=1.5, zorder=2)

r, p = stats.pearsonr(spend_corr['ProfDev_pp'], spend_corr['avg_turnover'])
ax.set_title(f'Professional Development Spending vs Teacher Turnover\n'
             f'r = {r:.2f}, p = {p:.3f}', fontsize=14, fontweight='bold')
ax.set_xlabel('Prof Dev Spending per Pupil ($)', fontsize=12)
ax.set_ylabel('Avg Teacher Turnover (%)', fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Cohort Reading Growth

Longitudinal reading proficiency for the same cohort from Grade 3 to Grade 6 — a growth view that complements the point-in-time assessment scores.

In [ ]:
g3 = cohort[cohort['grade'] == 3][['school', 'subgroup', 'cohort_pct_proficient_reading']]\
    .rename(columns={'cohort_pct_proficient_reading': 'Grade_3'})
g6 = cohort[cohort['grade'] == 6][['school', 'subgroup', 'cohort_pct_proficient_reading']]\
    .rename(columns={'cohort_pct_proficient_reading': 'Grade_6'})
cohort_summary = g3.merge(g6, on=['school', 'subgroup'])
cohort_summary['Movement'] = (cohort_summary['Grade_6'] - cohort_summary['Grade_3']).round(1)

print("=== COHORT READING GROWTH (ALL subgroup), Grade 3 -> Grade 6 ===")
print(cohort_summary[cohort_summary['subgroup'] == 'ALL']
      .sort_values('Movement', ascending=False).to_string(index=False))

## 13. Export for Tableau

Twelve clean, join-ready CSVs written to `data/clean/`, each with a consistent `school_name` key. These are the source files behind the [Pinehill Analytics Portal on Tableau Public](https://public.tableau.com/views/tabschool/Dashboard1).

In [ ]:
# ── 1. Student level: demographics + attendance ──
student_level = merged[['PERM_ID', 'school_name', 'GR', 'GENDER', 'RACE1',
                        'HISP', 'EL_STAT', 'SPED', 'FRL', 'SED',
                        'absence_rate', 'TARDIES', 'chronically_absent']].copy()
student_level.to_csv(f'{OUTPUT_DIR}/01_student_demographics_attendance.csv', index=False)

# ── 2. School-level master summary ──
school_summary = master_table.merge(
    positions_avg[['school_name', 'avg_turnover', 'avg_filled_fte',
                   'avg_vacant_fte', 'avg_days_to_fill', 'vacant_per_100']],
    on='school_name')
school_summary = school_summary.merge(
    spend_pivot.drop(columns='ProfDev_pp'), on='school_name')
school_summary = school_summary.merge(
    funding_need[['school_name', 'el_or_sed', 'Title I Federal',
                  'LCFF Supplemental', 'titleI_per_need', 'lcff_per_need',
                  'recommended_titleI', 'recommended_lcff']], on='school_name')
school_summary['total_intervention_funding'] = (
    school_summary['Title I Federal'] + school_summary['LCFF Supplemental'])
school_summary['recommended_total_funding'] = (
    school_summary['recommended_titleI'] + school_summary['recommended_lcff'])
school_summary['funding_difference'] = (
    school_summary['recommended_total_funding'] -
    school_summary['total_intervention_funding'])
school_summary.to_csv(f'{OUTPUT_DIR}/02_school_summary.csv', index=False)

# ── 3-4. Assessment (ALL subgroup by year; all subgroups) ──
assess_cols = ['school_name', 'school_year', 'grade', 'subgroup',
               'ela_pct_proficient', 'math_pct_proficient']
assessment_clean = assessment[assessment['subgroup'] == 'ALL'][assess_cols].copy()
assessment_clean['avg_score'] = ((assessment_clean['ela_pct_proficient'] +
                                  assessment_clean['math_pct_proficient']) / 2).round(1)
assessment_clean.to_csv(f'{OUTPUT_DIR}/03_assessment_by_year.csv', index=False)
assessment[assess_cols].to_csv(f'{OUTPUT_DIR}/04_assessment_all_subgroups.csv', index=False)

# ── 5-6. Cohort reading (raw + growth summary) ──
cohort.to_csv(f'{OUTPUT_DIR}/05_cohort_reading.csv', index=False)
cohort_export = cohort_summary.copy()
cohort_export['movement_direction'] = cohort_export['Movement'].apply(
    lambda x: 'Improving' if x > 2 else 'Declining' if x < -2 else 'Flat')
cohort_export.to_csv(f'{OUTPUT_DIR}/06_cohort_growth_summary.csv', index=False)

# ── 7. Staffing by year ──
staffing_yoy = positions[['SiteName', 'FiscalYear', 'BudgetedFTE', 'FilledFTE',
                          'VacantFTE', 'TeacherTurnoverPct',
                          'AvgDaysToFillVacancy']].copy()
staffing_yoy.columns = ['school_name', 'fiscal_year', 'budgeted_fte', 'filled_fte',
                        'vacant_fte', 'turnover_pct', 'avg_days_to_fill']
staffing_yoy['vacant_per_100'] = (
    staffing_yoy['vacant_fte'] /
    (staffing_yoy['filled_fte'] + staffing_yoy['vacant_fte']) * 100).round(2)
staffing_yoy.to_csv(f'{OUTPUT_DIR}/07_staffing_by_year.csv', index=False)

# ── 8. Budget by year & category ──
budget_detail = gl_final[['school_name', 'FiscalYear', 'AccountDescription',
                          'Budget', 'Actuals', 'Encumbrance', 'Balance']].copy()
budget_detail.columns = ['school_name', 'fiscal_year', 'category', 'budget',
                         'actuals', 'encumbrance', 'balance']
budget_detail = budget_detail.merge(
    schools_ref[['school_name', 'enrollment_2025_26']], on='school_name')
budget_detail['actuals_pp'] = (budget_detail['actuals'] /
                               budget_detail['enrollment_2025_26']).round(0)
budget_detail['budget_pp'] = (budget_detail['budget'] /
                              budget_detail['enrollment_2025_26']).round(0)
budget_detail['spend_rate_pct'] = (budget_detail['actuals'] /
                                   budget_detail['budget'] * 100).round(2)
budget_detail.to_csv(f'{OUTPUT_DIR}/08_budget_by_year_category.csv', index=False)

# ── 9. Course grades (deduped) ──
grades_clean.to_csv(f'{OUTPUT_DIR}/09_course_grades_df_rates.csv', index=False)

# ── 10. Funding equity ──
funding_export = funding_need.copy()
funding_export['total_current'] = (
    funding_export['Title I Federal'] + funding_export['LCFF Supplemental'])
funding_export['total_recommended'] = (
    funding_export['recommended_titleI'] + funding_export['recommended_lcff'])
funding_export['total_difference'] = (
    funding_export['total_recommended'] - funding_export['total_current'])
funding_export['pct_need'] = (funding_export['el_or_sed'] /
                              funding_export['total_students'] * 100).round(1)
funding_export.to_csv(f'{OUTPUT_DIR}/10_funding_equity.csv', index=False)

# ── 11. Attendance by school ──
attend_school.to_csv(f'{OUTPUT_DIR}/11_attendance_by_school.csv', index=False)

# ── 12. Correlation summary (computed in Section 9) ──
corr_summary.to_csv(f'{OUTPUT_DIR}/12_correlation_summary.csv', index=False)

print("=== EXPORTS COMPLETE ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size:,} bytes)")

## 14. Limitations & Conclusions

**Key findings**
1. EL% shows the strongest negative relationship with achievement; combined demographic and attendance variables explain ~94% of school-level score variation.
2. Intervention funding per student in need varies roughly 3x across schools and is not proportional to need — a budget-neutral, need-based reallocation formula is recommended.
3. Teacher turnover is significantly associated with chronic absenteeism, and professional development spending is the only budget category significantly associated with lower turnover — the highest-turnover schools invest the least in it.

**Limitations**
- **Small sample:** n = 12 schools. School-level aggregation inflates correlations relative to student-level data; findings are directional signals, not causal estimates.
- **Collinearity:** EL% and Free Lunch% correlate at r ≈ 0.96 and cannot be separated at this grain.
- **Short history:** only two years of staffing and budget data; limited budget variation (~95% spend rate district-wide).
- **Join constraints:** student-level joins were only possible within SIS data; finance, HR, and assessment systems merge at the school level.
- **Correlation ≠ causation** throughout. Findings should guide further investigation — student-level assessment data, teacher experience/credentials, and program participation data would strengthen all of these analyses.